# Bivariate analysis of categorical variables and the chi-square test

This notebook presents a bivariate analysis of categorical variables, specifically examining the relationship between individuals' countrys of origin, birth years and genders. A chi-square test is employed to determine if there is a statistically significant association between these factors over time. 

The aim is to understand how the geographical distribution of the population has changed over time and whether there is a significant over-representation of female astronomers/physicists in certain countrys.


In [ ]:
import pandas as pd

import scipy.stats as stats
import statsmodels.api as sm

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import numpy as np
import seaborn as sns

In [ ]:
import pprint
import csv
import sys

import time
import datetime
from dateutil import parser
from shutil import copyfile


In [ ]:
import warnings
warnings.filterwarnings('ignore')


## Create a dataframe with the data to be analysed

We use in this notebook the data produced in the da2 chapter, i.e. a list of persons with birth year, gender, place of birth, world country of birth

In [ ]:
csv_address='da_data/da3-birthYear-gender-region-country.csv'
df_p = pd.read_csv(csv_address)
df_p.head()

In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_p.info()

### Verify variables: gender

In [ ]:
### Group and count
# We observe some dispersion that requires grouping the categories of the variable
df_gender = df_p.groupby('gender').size()
df_gender = pd.DataFrame(df_gender.sort_values(ascending = False))
df_gender.columns=['number']
print(df_gender)

## Distribution of countries of birth

In [ ]:
### Group and count
# We observe some dispersion that requires grouping the categories of the variable
df_country = df_p.groupby('NAME_ENGL').size()
df_country = pd.DataFrame(df_country.sort_values(ascending = False))
df_country.columns=['number']
print(df_country.iloc[:50])
#print(df_country[(df_country.number < 21) & (df_country.number >5)])



In [ ]:
### Example of values
print(len(df_p[df_p.NAME_ENGL=='Chile']))
df_p[df_p.NAME_ENGL=='Chile'].head(3)

In [ ]:
### We define a function that codes and aggregates the values in order to avoid dispersion, 
# which would make them difficult to analyse.

def codeCountry(country: str):
    if 'Russian Federation' in country \
        or 'Kazakhstan' in country \
        or 'Georgia' in country \
        or 'Uzbekistan' in country \
        or 'Armenia' in country \
        or 'Azerbaijan' in country:
        output='Russian Federation'
    elif 'Estonia' in country \
        or 'Latvia' in country \
        or 'Finland' in country \
        or 'Belarus' in country \
        or 'Lithuania' in country:
        output='Baltic States Fin. Belar.'
    elif 'Moldova' in country \
        or 'Serbia' in country \
        or 'Slovenia' in country \
        or 'Slovakia' in country \
        or 'Romania' in country \
        or 'Croatia' in country \
        or 'Bulgaria' in country:
        output='Central Europe'
    elif 'Norway' in country \
        or 'Denmark' in country \
        or 'Sweden' in country:
        output='Scandinavia'
    elif 'Argentina' in country \
        or 'Brazil' in country \
        or 'Chile' in country:
        output='South America'    
    elif 'Spain' in country \
        or 'Portugal' in country:
        output='Spain Port.'    
    elif 'Belgium' in country \
        or 'Netherlands' in country:
        output='Belgium Netherl.'   
    elif 'Australia' in country \
        or 'New Zealand' in country:
        output='Australia New Z.'      
    elif 'Canada' in country \
        or 'United States' in country:
        output='United States Can.'   
    elif 'Austria' in country \
        or 'Hungary' in country:
        output='Austria Hungary'       
    else:
        output=country
    return output                   

In [ ]:
### Test the function
r='Belgium'
#r='Latvia'
codeCountry(r)

In [ ]:
### Create a new column with the coded values
df_p['coded_country']=df_p.NAME_ENGL.apply(lambda x : codeCountry(x))

In [ ]:
### Example
df_p.iloc[31:35]

In [ ]:
### Group and count
df_country = df_p.groupby('coded_country').size()
df_country = pd.DataFrame(df_country.sort_values(ascending = False))
#df_contCode.reset_index(inplace=True)
df_country.columns=['number']
df_country.iloc[:50]

In [ ]:
### countries with more than 200 persons (Israel and Australia are excluded)
df_country_200 = df_country[df_country.number > 200]
print(len(df_country_200))

In [ ]:
df_country_200

In [ ]:
lc200=df_country_200.index.to_list()
lc200

### Limit the analysis to persons with a coded birth country

In [ ]:
df_pa = df_p[df_p.coded_country.isin(lc200)]

In [ ]:
print('df_p:', len(df_p), '/ dfpa:', len(df_pa))
df_pa.iloc[31:35]

### Transform birth years to periods of activity years

In [ ]:
### Create imputed activity year
df_pa['activityYear'] = df_pa.birthYear.apply(lambda x : int(x)+45)

In [ ]:
df_pa = df_pa.drop([ 'periodsActivity'], axis=1)

In [ ]:
### Create list of 25 years periods

yr = df_pa.activityYear

l_25 = list(range(min(yr), max(yr)+ 25, 25))
print(l_25[:5],l_25[-5:], len(l_25)-1)

In [ ]:
### fonction pd.cut : https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.cut.html
# A new column is added containing the period, based on the previous list and the year value

df_pa['periodsActivity'] = pd.cut(df_pa['activityYear'], l_25, right=False)

### Rewrite the added code to make it more readable
df_pa['periodsActivity'] = df_pa['periodsActivity'].apply(lambda x : str(int(x.left))+'-'+ str(int(x.right)-1))

In [ ]:
# Inspection
df_pa.head(3)

In [ ]:
### Distribution of activities by 25 years periods
activities_per = pd.DataFrame(df_pa.groupby(by='periodsActivity').size())
activities_per.columns=['number']
print(activities_per)

#### Prepare the data for the AFC, cf. challenge da4

In [ ]:
file_address='da_data/da4-AFC.csv'
df_pa.to_csv(file_address, index=False)

## Bivariate analysis coded countries and activity periods

* A contingency table organises data to show the frequency of two or more categorical variables arranged in rows and columns, revealing possible relationships between them.
* Frequency counts are produced by calculating how often each combination of categories occurs in the dataset.

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
per_vs_country=pd.crosstab(df_pa.periodsActivity, df_pa.coded_country, margins=True)

## display all columns
pd.set_option('display.max_columns', None)

per_vs_country.iloc[2:]

In [ ]:
observed = per_vs_country.iloc[2:-1, :-1 ]
observed

In [ ]:
### Calculation of parameters for the chi-square test
statistic, p, dof, expected = stats.chi2_contingency(observed)


In [ ]:
dfe=pd.DataFrame(expected.round(0))
dfe.columns=observed.columns
dfe.index=observed.index
dfe

In [ ]:
## Degrees of freedom and Chi-square

print('Degrees of freedom:', dof, '; Chi-square value:', statistic.round(2))


In [ ]:
### P-value
print('p-value :', p)

In [ ]:
### Cramér's V (normalized phi) coefficient
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.association.html
vc = stats.contingency.association(observed, method='cramer')
print('Cramèr\'s V:', vc)

In [ ]:
### Get signed resitduals using statmodels (sm)

# 1. Create the Table object directly from your data
table = sm.stats.Table(observed)

# 2. Get Adjusted Residuals instantly (no manual formula needed)
adjusted_resids = table.standardized_resids
adjusted_resids.round(1)

In [ ]:
### Plot the residuals

# 3. Plot
fig, ax = plt.subplots(1,1,figsize=(26,8))         


# Create heatmap
sns.heatmap(
    adjusted_resids.round(1), 
    annot=True,            # Use boolean True to annotate with data values
    cmap="coolwarm", 
    linewidths=.5, 
    ax=ax,
    cbar_kws={'label': 'Residuals'}
)
# 3. Fix Label Rotation (Safe Method)
# This rotates existing ticks without risking a count mismatch
ax.set_xticklabels(ax.get_xticklabels(), rotation=80, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=20, va='center')
ax.set_title("Adjusted Residual", fontsize=12)

# ax.set_title("Heatmap of Adjusted Residuals (via statsmodels)")
plt.tight_layout()
plt.show()

In [ ]:
## Compare with number
per_vs_country.iloc[2:]

## Bivariate analysis genders

In [ ]:
gender_vs_country=pd.crosstab(df_pa.gender, df_pa.coded_country, margins=True)
gender_vs_country

In [ ]:
### Take the margins away
observed = gender_vs_country.iloc[:-1, :-1 ]
### Calcul des paramètres pour le test du Chi-2
statistic, p, dof, expected = stats.chi2_contingency(observed)
print('Chi2 :', statistic, ', dof :',dof)
print('p-value :', p)

In [ ]:
### Cramér's phi coefficient
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.association.html
vc = stats.contingency.association(observed, method='cramer')
print('Cramèr\'s V:', vc)

In [ ]:
# 1. Create the Table object directly from your data
table = sm.stats.Table(observed)

# 2. Get Adjusted Residuals instantly (no manual formula needed)
adjusted_resids = table.standardized_resids
# 3. Plot
fig, ax = plt.subplots(figsize=(20,2))         
# Sample figsize in inches
ax = sns.heatmap(adjusted_resids, annot=adjusted_resids, cmap="coolwarm", linewidths=.5, ax=ax)
labels = adjusted_resids.index
p = ax.set_yticklabels(labels, rotation=30)
labels_cols = adjusted_resids.columns
p = ax.set_xticklabels(labels_cols, rotation=80)

ax.set_title("Heatmap of Adjusted Residuals (via statsmodels)")
plt.show()


We can observe that women are more present in some countries. The difference is statistically relevant although weak: Cramer's V = 0.116

## Bivariate analysis genders+generations vs countrys

In [ ]:
def code_gender_period(gender: str):
    if gender == 'female':
        output='f'
    else:
        output='m'
    return output    

In [ ]:
df_pa['per_gender']= df_pa.apply(lambda x: x.periodsActivity +'_'+ code_gender_period(x.gender), axis=1)
df_pa.head(2)

In [ ]:
per_gender_vs_country=pd.crosstab(df_pa.per_gender, df_pa.coded_country, margins=True)
per_gender_vs_country.iloc[6:]

In [ ]:
### Only mor recent periods
observed = per_gender_vs_country.iloc[6:-1, :-1 ]

In [ ]:
### For significant chi-2 test minimal number in each cell of expected:5
# bias in the results
statistic, p, dof, expected = stats.chi2_contingency(observed)
dfe = pd.DataFrame(expected).round(0)
dfe.index = observed.index
dfe.columns = observed.columns
dfe

In [ ]:
print('Chi2 :', statistic, ', dof :',dof)
print('p-value :', p)

In [ ]:
### Cramér's phi coefficient
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.association.html
vc = stats.contingency.association(observed, method='cramer')
print('Cramèr\'s V:', vc)

In [ ]:
# 1. Create the Table object directly from your data
table = sm.stats.Table(observed)

# 2. Get Adjusted Residuals instantly (no manual formula needed)
adjusted_resids = table.standardized_resids
# 3. Plot
fig, ax = plt.subplots(figsize=(20,6))         
# Sample figsize in inches
ax = sns.heatmap(adjusted_resids, annot=adjusted_resids, cmap="coolwarm", linewidths=.5, ax=ax)
labels = adjusted_resids.index
p = ax.set_yticklabels(labels, rotation=30)
labels_cols = adjusted_resids.columns
p = ax.set_xticklabels(labels_cols, rotation=80)

ax.set_title("Heatmap of Adjusted Residuals (via statsmodels)")
plt.show()


In [ ]:
dfs = df_pa[(df_pa.per_gender.isin(['2001-2025_f']))&(df_pa.coded_country.isin(['South America', 'Spain Port.']))]
print(len(dfs))

In [ ]:
dfs.head(10)

### Inspected persons

* [Josefa Masegosa Gallego](http://www.wikidata.org/entity/Q19812690) (Spain)
* [Judith Desimoni](http://www.wikidata.org/entity/Q62072509) (Argentina)